# Episode 4 — BBSW swaps

Companion notebook for the video. We build a 3-month BBSW curve from illustrative quotes, look at the cash flows of a quarterly swap, derive the par swap rate by hand, and price an off-market swap and its DV01. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Illustrative data.** The quotes in `quotes_illustrative.csv` are made up for teaching. They are shaped like a plausible AUD curve but are **not market prices** (BBSW is licensed ASX data). Educational material only, not investment advice.
>
> **One curve, on purpose.** In this episode the BBSW curve both forecasts BBSW and discounts. The market discounts on the AONIA curve instead (AFMA §3.6); Episode 5 makes that change and shows what it does.

1. Setup · 2. Conventions · 3. Quotes · 4. Bootstrap · 5. Cash flows of a 1-year swap · 6. The par rate by hand · 7. An off-market swap · 8. DV01 · 9. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""tenor,instrument,rate_pct
3M,BBSW,4.02
3x6,FRA,4.08
6x9,FRA,4.13
9x12,FRA,4.17
12x15,FRA,4.20
15x18,FRA,4.23
18x21,FRA,4.25
2Y,Swap,4.17
3Y,Swap,4.22
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-22"
quotes_file = "quotes_illustrative.csv"
output_json = "build/outputs.json"
notional = 100_000_000
offmarket_fixed_pct = 4.50

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
iso = lambda d: d.ISO()

cal = ql.Australia(ql.Australia.Settlement)
# QuantLib 1.43 misses NSW's additional Anzac Day holidays when 25 April falls on a weekend.
for d in [ql.Date(27, 4, 2026), ql.Date(26, 4, 2027)]:
    cal.addHoliday(d)

def bbsw3m(curve=ql.YieldTermStructureHandle()):
    # 3M BBSW: set on the first day of the period (no fixing lag), Modified Following, no end-of-month rule.
    # (ql.Bbsw3M() uses half-month modified following with end-of-month on, so we define our own.)
    return ql.IborIndex("BBSW3M", ql.Period(3, ql.Months), 0, ql.AUDCurrency(), cal,
                        ql.ModifiedFollowing, False, dc, curve)

out = {"meta": {
    "episode": 4, "valuation_date": valuation_date, "quantlib_version": ql.__version__,
    "quotes_label": "ILLUSTRATIVE - not market data",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__, "| valuation date", today)

## 2. Conventions

| Item | Convention | Source |
|---|---|---|
| Floating rate | 3-month BBSW for quarterly swaps (out to 3 years); 6-month BBSW for semi-annual swaps (4 years and longer) | AFMA §2.2, §2.3, §3.7 |
| Fixing | BBSW for each period is set on the first day of the period and paid at its end (in arrears) | AFMA §2.2; RBA Bulletin (June 2022) |
| Day count | ACT/365 Fixed on both legs | AFMA §3.7 |
| Start | next Sydney business day (T+1) | AFMA §5.2 |
| Dates | Modified Following, no end-of-month rule | AFMA §3.3 |
| Payment | on each period end date, both legs quarterly (or both semi-annual) | AFMA §2.2 |

In [ ]:
out["conventions"] = [
    {"item": "Floating rate", "convention": "3M BBSW (quarterly), to 3 years"},
    {"item": "From 4 years", "convention": "6M BBSW, semi-annual"},
    {"item": "Fixing", "convention": "set at the start of each period"},
    {"item": "Payment", "convention": "at the end of each period"},
    {"item": "Day count", "convention": "ACT/365 Fixed, both legs"},
    {"item": "Start and dates", "convention": "T+1, Modified Following, no EOM"},
]
pd.DataFrame(out["conventions"])

## 3. Quotes (illustrative)

The 3-month BBSW fixing, FRAs on 3-month BBSW, and quarterly swaps. An FRA `3x6` covers the three months starting three months from now.

In [ ]:
quotes = pd.read_csv(quotes_file)
out["quotes"] = quotes.to_dict("records")
quotes

## 4. Bootstrap a single BBSW curve

Each quote becomes a rate helper; one log-linear discount curve is solved so that every helper reprices to its quote.

In [ ]:
def make_helpers(index):
    qs, hs = {}, []
    for row in quotes.itertuples():
        q = ql.SimpleQuote(row.rate_pct / 100)
        qs[row.tenor] = q
        if row.instrument == "BBSW":
            h = ql.DepositRateHelper(ql.QuoteHandle(q), index)
        elif row.instrument == "FRA":
            months_to_start = int(row.tenor.split("x")[0])
            h = ql.FraRateHelper(ql.QuoteHandle(q), months_to_start, index)
        else:  # quarterly swap against 3M BBSW, starting T+1
            h = ql.SwapRateHelper(ql.QuoteHandle(q), ql.Period(row.tenor), cal, ql.Quarterly,
                                  ql.ModifiedFollowing, dc, index, ql.QuoteHandle(), ql.Period(0, ql.Days),
                                  ql.YieldTermStructureHandle(), 1)
        hs.append((row.tenor, row.instrument, h))
    return qs, hs

quote_handles, helpers = make_helpers(bbsw3m())
curve = ql.PiecewiseLogLinearDiscount(today, [h for *_, h in helpers], dc)
curve.enableExtrapolation()
curve_handle = ql.YieldTermStructureHandle(curve)
curve.nodes()  # bootstrap now

rep = pd.DataFrame([{"tenor": t, "instrument": k, "quote_pct": quote_handles[t].value() * 100,
                     "implied_pct": h.impliedQuote() * 100, "pillar_date": iso(h.pillarDate()),
                     "discount_factor": curve.discount(h.pillarDate())} for t, k, h in helpers])
rep["error_bp"] = (rep.implied_pct - rep.quote_pct) * 100
out["pillars"] = rep.to_dict("records")
out["reprice_max_abs_error_bp"] = float(rep.error_bp.abs().max())
rep

The 3-month forward BBSW rates this curve implies, month by month, with the FRA quotes on top.

In [ ]:
index = bbsw3m(curve_handle)
spot = cal.advance(today, 1, ql.Days)
grid = []
for m in range(0, 34):  # forward windows that end within the 3-year curve
    d0 = cal.advance(today, m, ql.Months)
    d1 = cal.advance(d0, 3, ql.Months, ql.ModifiedFollowing, False)
    grid.append({"t_years": dc.yearFraction(today, d0), "fwd3m_pct": curve.forwardRate(d0, d1, dc, ql.Simple).rate() * 100})
fra = quotes[quotes.instrument.isin(["BBSW", "FRA"])].copy()
fra["t_years"] = [dc.yearFraction(today, cal.advance(today, int(t.split("x")[0]) if "x" in t else 0, ql.Months)) for t in fra.tenor]
out["forwards"] = {"grid": grid, "fra_t": fra.t_years.tolist(), "fra_pct": fra.rate_pct.tolist(),
                   "ticks": [{"x": i / 2, "label": lab} for i, lab in enumerate(["0", "6M", "1Y", "18M", "2Y", "30M"])]}
g = pd.DataFrame(grid)
plt.figure(figsize=(9, 3.5))
plt.plot(g.t_years, g.fwd3m_pct, label="3M forward BBSW")
plt.scatter(fra.t_years, fra.rate_pct, color="k", zorder=3, label="BBSW / FRA quotes")
plt.grid(alpha=.3); plt.legend(); plt.ylabel("%"); plt.xlabel("years")

## 5. Cash flows of a 1-year quarterly swap

Receive fixed at the 1-year par rate. Each floating coupon uses the 3-month BBSW rate set on the first day of its period and is paid at the end of the period. With one curve and no payment lag, the floating rate for each period is the forward rate over that period.

In [ ]:
def make_swap(tenor, fixed_rate, receive=True, forecast=curve_handle, discount=curve_handle):
    idx = bbsw3m(forecast)
    end = cal.advance(spot, ql.Period(tenor), ql.ModifiedFollowing, False)
    sched = ql.Schedule(spot, end, ql.Period(ql.Quarterly), cal, ql.ModifiedFollowing, ql.ModifiedFollowing,
                        ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, idx, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

par_1y = make_swap("1Y", 0.04).fairRate()
sw1 = make_swap("1Y", par_1y)
rows = []
for f, fl in zip(sw1.fixedLeg(), sw1.floatingLeg()):
    fc, c = ql.as_coupon(f), ql.as_floating_rate_coupon(fl)
    rows.append({"fixing_date": iso(c.fixingDate()), "accrual_start": iso(c.accrualStartDate()),
                 "accrual_end": iso(c.accrualEndDate()), "payment_date": iso(fl.date()), "days": c.accrualDays(),
                 "float_rate_pct": c.indexFixing() * 100, "fixed_amount": f.amount(), "float_amount": fl.amount(),
                 "df": curve.discount(fl.date())})
cf = pd.DataFrame(rows)
out["cashflows_1y"] = {"notional": notional, "fixed_rate_pct": par_1y * 100, "rows": rows, "npv": sw1.NPV(),
                       "fixing_is_start": bool((cf.fixing_date == cf.accrual_start).all()),
                       "paid_at_end": bool((cf.payment_date == cf.accrual_end).all())}
cf

## 6. The par rate by hand

The par rate $S$ makes the two legs equal:

$$S \sum_i \tau_i P(t_i) = \sum_i \tau_i F_i P(t_i)$$

The left-hand sum is the **annuity** $A$: the value of receiving 1 per year, paid quarterly. With one curve, each forward is $F_i = \big(P(t_{i-1})/P(t_i) - 1\big)/\tau_i$, so the floating leg telescopes to $P(t_0) - P(t_n)$:

$$S = \frac{P(t_0) - P(t_n)}{A}$$

Check it on the 3-year swap, which is one of our quotes.

In [ ]:
sw3 = make_swap("3Y", quote_handles["3Y"].value())
pay_dates = [c.date() for c in sw3.fixedLeg()]
taus = [ql.as_coupon(c).accrualPeriod() for c in sw3.fixedLeg()]
annuity = sum(t * curve.discount(d) for t, d in zip(taus, pay_dates))
p0, pn = curve.discount(spot), curve.discount(pay_dates[-1])
par = {"tenor": "3Y", "n_periods": len(pay_dates), "p_start": p0, "p_end": pn, "annuity": annuity,
       "float_leg_per_unit": p0 - pn, "par_hand_pct": (p0 - pn) / annuity * 100,
       "par_ql_pct": sw3.fairRate() * 100, "quote_pct": quote_handles["3Y"].value() * 100}
par["abs_diff_bp"] = abs(par["par_hand_pct"] - par["par_ql_pct"]) * 100
par["annuity_ql"] = abs(sw3.fixedLegBPS()) / (notional * 1e-4)  # fixedLegBPS: value of 1bp on the fixed leg
out["par"] = par
pd.Series(par)

## 7. An off-market swap

Receive a fixed rate $K$ that is not the par rate. The floating leg is worth the same as before, so the whole value comes from the rate difference:

$$\text{NPV}_\text{receiver} = N\,(K - S)\,A$$

In [ ]:
K = offmarket_fixed_pct / 100
sw_off = make_swap("3Y", K)
off = {"fixed_rate_pct": offmarket_fixed_pct, "par_pct": par["par_ql_pct"], "annuity": annuity,
       "npv_hand": notional * (K - par["par_ql_pct"] / 100) * annuity, "npv_ql": sw_off.NPV()}
off["abs_diff"] = abs(off["npv_hand"] - off["npv_ql"])
out["offmarket"] = off
pd.Series(off)

## 8. DV01

Bump every quote by one basis point, rebuild the curve, and reprice the 3-year receiver at par. Compare with the *PV01* of the fixed leg, $N \cdot A \cdot 0.0001$: for a par swap they are close, because the annuity is what a 1bp change in the swap rate is worth.

In [ ]:
def npv_after_shift(swap, bp):
    for q in quote_handles.values():
        q.setValue(q.value() + bp / 1e4)
    v = swap.NPV()
    for q in quote_handles.values():
        q.setValue(q.value() - bp / 1e4)
    return v

sw3_par = make_swap("3Y", par["par_ql_pct"] / 100)
up, down = npv_after_shift(sw3_par, +1), npv_after_shift(sw3_par, -1)
out["dv01"] = {"tenor": "3Y", "notional": notional, "npv_up_1bp": up, "npv_down_1bp": down,
               "dv01": (down - up) / 2, "pv01": notional * annuity * 1e-4}
out["dv01"]["ratio"] = out["dv01"]["dv01"] / out["dv01"]["pv01"]
pd.Series(out["dv01"])

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")